In [65]:
import os,json,markdown,re,requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown,display,update_display
from reportlab.platypus import Paragraph,SimpleDocTemplate
from reportlab.lib.styles import getSampleStyleSheet
import gradio as ui

load_dotenv(override=True)

base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv("GOOGLE_API_KEY")
model = "gemini-3.6-flash"

myAI = OpenAI(base_url=base_url,api_key=api_key)

In [66]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [67]:
def fetch_website_content(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content,"html.parser")
    title = soup.title.string if soup.title else "None found"
    if soup.body :
        for irrelevant in soup.body(["script","style","img","input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n",strip=True)
    else:
        text = ""
    return text[:2000]

In [68]:
def fetch_website_links(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content,"html.parser")
    links = []
    for link in soup.find_all("a"):
        links.append(link.get("href"))
    return links

In [69]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [70]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.
    
        Links (some might be relative links):
        """
    links = fetch_website_links(url)
    user_prompt+= "\n".join(links)
    return user_prompt

In [71]:
def select_relevant_links(url):
    response = myAI.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_prompt(url)}
            ],
        response_format={'type':'json_object'},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [72]:
def fetch_page_and_relevant_links(url):
    contents = fetch_website_content(url)
    relevant = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_content(link["url"])

    return result

In [73]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
Give the brochure in a format that is suitable for printing as a PDF, with headings, subheadings, and bullet points where appropriate.
"""

In [74]:
def get_brochure_user_prompt(company_name,url):
    user_prompt = f"""
        You are looking at a company called: {company_name}
        Here are the contents of its landing page and other relevant pages;
        use this information to build a short brochure of the company in markdown without code blocks.\n\n
        """
    user_prompt +=fetch_page_and_relevant_links(url)
    user_prompt = user_prompt[:5000] 
    return user_prompt

In [86]:
def create_brochure(message,history):

    try:
        company_name, url = [x.strip() for x in message.split(" ", 1)]
    except ValueError:
        return "❌ Please enter in this format:\n\nCompany Name  https://example.com"

    download_dir = os.path.expanduser("~/Downloads")
    os.makedirs(download_dir, exist_ok=True)


    stream = myAI.chat.completions.create(
        model=model,
        messages=[{'role':'system','content':brochure_system_prompt},{'role':'user','content':get_brochure_user_prompt(company_name,url)}],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

    html = markdown.markdown(response)
    html = re.sub(r'</?(ul|ol|li|h1|h2|h3|h4|h5|h6)>', '', html)

    pdf_path = os.path.join(download_dir,f"{company_name}_Brochure.pdf" )
    pdf = SimpleDocTemplate(pdf_path)
    styles = getSampleStyleSheet()

    story = []
    for line in html.split('\n'):
        if line.split():
            story.append(Paragraph(line,styles["BodyText"]))
    pdf.build(story)
    
    return f"✅ PDF saved as {company_name}_Brochure.pdf \n ✅ PDF successfully saved to: `{pdf_path}`"


In [85]:
demo = ui.ChatInterface(
    fn = create_brochure,
    title="I can Create Brochure for You",
    description="Just Send me a company_name and its link"
)

demo.launch(share=True,inline=False,inbrowser=True)

* Running on local URL:  http://127.0.0.1:7873
* Running on public URL: https://9d5c15296e6884f55b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\gradio\queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\gradio\route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\gradio\blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\gradio\blocks.py", line 1688, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jatin\AppD